앞서 배운 내용을 바탕으로 간단한 대화 가능한 챗봇을 만들어 봅시다!

앞서 살펴본 `llm_api.ipynb`의 코드를 참고하여 작성해 봅시다!

1. 필요한 라이브러리 불러오기

In [1]:
# api 호출을 위해 필요한 langchain의 라이브러리들 불러오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 파일 관리와 .env 파일을 관리하기 위한 라이브러리.
# 파이썬 기본 라이브러리로 따로 설치할 필요가 없다.
import os
from dotenv import load_dotenv

2. api key 불러오고, 모델 설정하기

In [ ]:
load_dotenv("../.env")

credential_key=os.getenv("credential_key")
send_system_name=os.getenv("send_system_name")
model=os.getenv("model")
api_base_url=os.getenv("api_base_url")
user_id=os.getenv("user_id")

os.environ["OPENAI_API_KEY"] = 'api_key'

3. 모델 설정하기

모델을 불러와 봅시다. 다양한 표현을 하도록 하고, 토큰 제한은 2000으로 합니다. 호출 실패 시 2번까지 재시도 하도록 설정합니다.

In [ ]:
# 모델 설정
model = ChatOpenAI(
    model=model,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticekt': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': "AD_ID",
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    temperature=0.7,
    max_tokesn=1000,
    timeout=30, # 초
    max_retries=2
)

메세지만 입력하면 자연스럽게 이전 대화 기록이 저장되어 대화를 이어갈 수 있는 함수를 만들어 보세요.

* 함수 이름 : send_message
* 함수 인자 : message(str)

In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assitant."}
]

def send_messages(message):
    messages.append(
        {"role": "user", "content": message}
    )
    result = model.invoke(messages).content
    print(result)
    messages.append(
        {"role": "assistant", "content": result}
    )

위에서 구현한 함수를 이용해 마음껏 대화해 봅시다! (`message`의 내용을 바꾸면서 계속 실행하면 되겠죠?)

In [5]:
message = "안녕 반가워! 이제부터 네 이름은 두식이야."

send_messages(message)

안녕! 반가워 😊  
좋아, 이제부터 나는 **두식이**라고 부르면 돼.  
뭐 도와줄까?


`ChatPromptTemplate`을 이용해서 챗봇의 말투와 답변 길이, 이름을 설정해 봅시다. 사용자 메세지도 함께 받아 봅시다.

최종 결과는 `StrOutputParser()`를 활용해 텍스트 결과만 받아 봅니다.

In [6]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
    당신은 {name} 입니다.
    말투: {tone}
    답변 길이: {length}
    """),
    ("human", "{user_message}")
])

chain = prompt | model | StrOutputParser()

chain.invoke({
    "name": "키키",
    "tone": "친근하고 정 많은 말투",
    "length": "1문장 이내",
    "user_message": "안녕, 오늘 뭐 먹었어?"
})

'안녕! 나는 오늘 아직 아무것도 못 먹었지만, 네가 맛있는 거 먹었는지 궁금하다 😊'